In [1]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Dict, Optional
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime
import pickle
import threading
import urllib.request

# -------------------------------------------------------------
# 1. تدريب وتجهيز المودل (ML Model Training)
# -------------------------------------------------------------

dates = pd.date_range(
    start="2026-01-01",
    end="2026-12-31 23:00:00",
    freq="h"
)

df = pd.DataFrame({"timestamp": dates})

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek

df["is_weekend"] = df["day_of_week"].apply(
    lambda x: 1 if x in [4, 5] else 0
)


def simulate_traffic(row):
    base_cars = 40

    # Evening peak
    if 16 <= row["hour"] <= 21:
        base_cars += 80

    # Morning peak
    elif 7 <= row["hour"] <= 10:
        base_cars += 40

    # Low traffic during early morning
    elif 1 <= row["hour"] <= 5:
        base_cars -= 25

    # Weekend
    if row["is_weekend"]:
        base_cars += 50

    cars = max(
        5,
        int(base_cars + np.random.normal(0, 10))
    )

    wait_minutes = int(
        cars * 0.4 + np.random.normal(0, 3)
    )

    return max(5, wait_minutes)


df["wait_minutes"] = df.apply(
    simulate_traffic,
    axis=1
)

X = df[
    [
        "hour",
        "day_of_week",
        "is_weekend"
    ]
]

y = df["wait_minutes"]


# Random Forest Model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)


# Save model
with open("traffic_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("✅ تم تدريب المودل وحفظ traffic_model.pkl")


# -------------------------------------------------------------
# 2. منطق الجاهزية والتنبؤ والتوصية (Core Engine)
# -------------------------------------------------------------

def evaluate_readiness(
    travel_date_str: str,
    documents: dict,
    travel_mode: str = "car"
):

    travel_date = datetime.strptime(
        travel_date_str,
        "%Y-%m-%d"
    ).date()


    # ---------------------------------------------------------
    # Document Rules
    # ---------------------------------------------------------

    if travel_mode.lower() == "car":

        doc_rules = {

            "passport": {
                "weight": 30,
                "label": "جواز السفر / الهوية الوطنية"
            },

            "insurance": {
                "weight": 25,
                "label": "تأمين المركبة"
            },

            "license": {
                "weight": 25,
                "label": "رخصة القيادة"
            },

            "vehicle_inspection": {
                "weight": 20,
                "label": "الفحص الدوري للمركبة"
            }
        }

    else:

        # السفر بالحافلة:
        # الاعتماد على وثيقة السفر الشخصية

        doc_rules = {

            "passport": {
                "weight": 100,
                "label": "جواز السفر / الهوية الوطنية"
            }
        }


    # ---------------------------------------------------------
    # Evaluate Documents
    # ---------------------------------------------------------

    total_score = 0

    missing_docs = []

    doc_details = {}


    for key, rule in doc_rules.items():

        expiry_str = documents.get(key)

        label = rule["label"]
        weight = rule["weight"]


        if expiry_str:

            try:

                exp_date = datetime.strptime(
                    expiry_str,
                    "%Y-%m-%d"
                ).date()


                # Document is valid
                if exp_date >= travel_date:

                    total_score += weight

                    doc_details[label] = "سارية"

                    continue


            except ValueError:
                pass


        # Missing or expired
        doc_details[label] = "منتهية أو ناقصة"

        missing_docs.append(
            f"{label} منتهية أو غير متوفرة"
        )


    # ---------------------------------------------------------
    # Readiness Level
    # ---------------------------------------------------------

    if total_score >= 85:

        level = "جاهزية عالية"

    elif total_score >= 60:

        level = "جاهزية متوسطة"

    else:

        level = "جاهزية منخفضة"


    # ---------------------------------------------------------
    # Requirements Count
    # ---------------------------------------------------------

    requirements_total = len(doc_details)

    requirements_met = sum(
        1
        for status in doc_details.values()
        if status == "سارية"
    )


    return (
        total_score,
        level,
        missing_docs,
        doc_details,
        requirements_total,
        requirements_met
    )


# -------------------------------------------------------------
# 3. Traffic Prediction
# -------------------------------------------------------------

def predict_traffic_and_best_time(
    travel_date_str: str,
    arrival_hour: int
):

    dt = datetime.strptime(
        travel_date_str,
        "%Y-%m-%d"
    )

    day_of_week = dt.weekday()

    is_weekend = (
        1 if day_of_week in [4, 5]
        else 0
    )


    # ---------------------------------------------------------
    # 24-Hour Forecast
    # ---------------------------------------------------------

    day_features = [
        [
            h,
            day_of_week,
            is_weekend
        ]

        for h in range(24)
    ]


    predictions = model.predict(
        day_features
    ).astype(int).tolist()


    wait_time_minutes = predictions[
        arrival_hour
    ]


    # ---------------------------------------------------------
    # Traffic Level
    # ---------------------------------------------------------

    if wait_time_minutes > 45:

        traffic_level = "high"

    elif wait_time_minutes > 25:

        traffic_level = "medium"

    else:

        traffic_level = "low"


    # ---------------------------------------------------------
    # Best Alternative Time ±3 Hours
    # ---------------------------------------------------------

    start = max(
        0,
        arrival_hour - 3
    )

    end = min(
        23,
        arrival_hour + 3
    )


    best_hour = min(
        range(start, end + 1),
        key=lambda h: predictions[h]
    )


    recommended_time = (
        f"{best_hour:02d}:00-"
        f"{(best_hour + 1):02d}:00"
    )


    return (
        traffic_level,
        wait_time_minutes,
        recommended_time,
        predictions
    )


# -------------------------------------------------------------
# 4. Generate Recommendation
# -------------------------------------------------------------

def generate_recommendation(
    missing_docs: list,
    traffic_level: str,
    recommended_time: str,
    arrival_time_str: str
):

    recs = []


    # ---------------------------------------------------------
    # Documents Recommendation
    # ---------------------------------------------------------

    if missing_docs:

        recs.append(
            "أكمل تجديد المتطلبات التالية: "
            f"({' ، '.join(missing_docs)})"
        )

    else:

        recs.append(
            "جميع وثائقك مكتملة وجاهزة للعبور"
        )


    # ---------------------------------------------------------
    # Traffic Recommendation
    # ---------------------------------------------------------

    if traffic_level == "high":

        recs.append(
            f"يتزامن وصولك عند الساعة "
            f"{arrival_time_str} مع ذروة ازدحام مرتفعة، "
            f"ويُفضل العبور بين "
            f"{recommended_time} لتفادي التأخير"
        )


    elif traffic_level == "medium":

        recs.append(
            f"حركة السير متوسطة عند الساعة "
            f"{arrival_time_str}، "
            f"وبإمكانك اختيار الفترة "
            f"{recommended_time} لعبور أسرع"
        )


    else:

        recs.append(
            f"توقيت وصولك عند الساعة "
            f"{arrival_time_str} ممتاز "
            f"وحركة السير خفيفة عند المنفذ"
        )


    return ". ".join(recs)


# -------------------------------------------------------------
# 5. إعداد الـ API
# -------------------------------------------------------------

nest_asyncio.apply()

app = FastAPI(
    title="Mohaya Core API"
)


# -------------------------------------------------------------
# CORS
# -------------------------------------------------------------

app.add_middleware(
    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=True,

    allow_methods=["*"],

    allow_headers=["*"],
)


# -------------------------------------------------------------
# 6. Request Model
# -------------------------------------------------------------

class ReadinessRequest(BaseModel):

    # Destination
    destination: str

    # Crossing information
    crossing: Optional[str] = None

    # Nationality
    nationality: Optional[str] = None


    # ---------------------------------------------------------
    # Personal / Document Information
    # ---------------------------------------------------------

    document_type: Optional[str] = None

    document_number: Optional[str] = None

    date_of_birth: Optional[str] = None

    document_issue_date: Optional[str] = None

    document_expiry_date: Optional[str] = None


    # ---------------------------------------------------------
    # Travel Information
    # ---------------------------------------------------------

    travel_date: str

    arrival_time: str

    travel_mode: str = "car"


    # ---------------------------------------------------------
    # Vehicle Information
    # ---------------------------------------------------------

    vehicle_type: Optional[str] = None

    vehicle_ownership: Optional[str] = None


    # ---------------------------------------------------------
    # Documents
    # ---------------------------------------------------------

    documents: Dict[
        str,
        Optional[str]
    ] = {}


# -------------------------------------------------------------
# 7. Main API Endpoint
# -------------------------------------------------------------

@app.post("/api/check-readiness")
def check_readiness(
    data: ReadinessRequest
):

    # ---------------------------------------------------------
    # Validate Arrival Time
    # ---------------------------------------------------------

    try:

        arrival_hour = int(
            data.arrival_time.split(":")[0]
        )

        if not 0 <= arrival_hour <= 23:

            raise ValueError


    except (ValueError, IndexError):

        return {
            "error": "Invalid arrival_time. Use HH:MM format."
        }


    # ---------------------------------------------------------
    # Readiness Evaluation
    # ---------------------------------------------------------

    (
        score,
        level,
        missing,
        doc_status,
        requirements_total,
        requirements_met
    ) = evaluate_readiness(

        data.travel_date,

        data.documents,

        data.travel_mode
    )


    # ---------------------------------------------------------
    # Traffic Prediction
    # ---------------------------------------------------------

    (
        traffic_level,
        wait_minutes,
        recommended_time,
        hourly_forecast

    ) = predict_traffic_and_best_time(

        data.travel_date,

        arrival_hour
    )


    # ---------------------------------------------------------
    # Generate Recommendation
    # ---------------------------------------------------------

    recommendation = generate_recommendation(

        missing,

        traffic_level,

        recommended_time,

        data.arrival_time
    )


    # ---------------------------------------------------------
    # API Response
    # ---------------------------------------------------------

    return {

        # Readiness
        "readiness_score": score,

        "readiness_level": level,

        "requirements_total": requirements_total,

        "requirements_met": requirements_met,


        # Traffic
        "traffic_level": traffic_level,

        "expected_wait_minutes": wait_minutes,

        "recommended_time": recommended_time,


        # Documents
        "missing_requirements": missing,

        "document_status": doc_status,


        # Chart Data
        "hourly_forecast": hourly_forecast,


        # Final Recommendation
        "recommendation": recommendation
    }


# -------------------------------------------------------------
# 8. Run FastAPI Server
# -------------------------------------------------------------

def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


threading.Thread(
    target=run_server,
    daemon=True
).start()


# -------------------------------------------------------------
# 9. LocalTunnel
# -------------------------------------------------------------

!npm install -g localtunnel > /dev/null 2>&1


external_ip = urllib.request.urlopen(
    "https://ipv4.icanhazip.com"
).read().decode("utf8").strip()


get_ipython().system_raw(
    "lt --port 8000 --subdomain mohaya-api &"
)


# -------------------------------------------------------------
# 10. Final Output
# -------------------------------------------------------------

print("🚀 السيرفر شغال وجاهز للربط!")

print(
    "🔑 الباسورد (Endpoint IP):",
    external_ip
)

print(
    "🔗 الرابط العام المباشر:",
    "https://mohaya-api.loca.lt/docs"
)

✅ تم تدريب المودل وحفظ traffic_model.pkl


INFO:     Started server process [519]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 السيرفر شغال وجاهز للربط!
🔑 الباسورد (Endpoint IP): 34.106.128.19
🔗 الرابط العام المباشر: https://mohaya-api.loca.lt/docs
